# 🌐 CareerLens AI — Phase 4: Dense Semantic Embeddings & KNN Benchmark Retrieval

**Objective**:
- Encode all 2,466 resumes into 384-dimensional dense semantic vectors using `sentence-transformers/all-MiniLM-L6-v2`.
- Compute Category Prototypes (centroids) for all 24 domains.
- Implement KNN Cosine Similarity search against the benchmark dataset.
- Clarify distinction: **Classification $\neq$ Semantic Similarity $\neq$ Career Recommendation**.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import json
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

print("Embedding libraries successfully loaded.")

Embedding libraries successfully loaded.


## 1. Load Pre-computed Embeddings & Model

In [2]:
emb_path = '../data/embeddings.npy' if os.path.exists('../data/embeddings.npy') else 'data/cv_embeddings.npy'
cv_embeddings = np.load(emb_path)

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')

print(f"Embeddings Matrix Shape: {cv_embeddings.shape}")
print(f"Dense Vector Dimension:  {cv_embeddings.shape[1]}")
print(f"Data Type:               {cv_embeddings.dtype}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Matrix Shape: (2466, 384)
Dense Vector Dimension:  384
Data Type:               float32


## 2. Category Prototypes (384-d Centroids)

In [3]:
proto_path = '../data/prototypes.json' if os.path.exists('../data/prototypes.json') else 'data/prototypes.json'
with open(proto_path, 'r') as f:
    prototypes = json.load(f)

print(f"Total Category Prototypes: {len(prototypes)}")
print("Categories:", list(prototypes.keys()))
print("Prototype Vector Dimension:", len(prototypes['INFORMATION-TECHNOLOGY']))

Total Category Prototypes: 24
Categories: ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']
Prototype Vector Dimension: 384


## 3. KNN Similar Resume Retrieval Demonstration

In [4]:
meta_path = '../data/cv_metadata.json' if os.path.exists('../data/cv_metadata.json') else 'data/cv_metadata.json'
with open(meta_path, 'r') as f:
    cv_metadata = json.load(f)

query_text = "Senior Python Developer with 5 years experience building FastAPI and Django microservices, SQL databases, and Docker pipelines."
query_emb = embedder.encode(query_text, convert_to_tensor=True)

cos_scores = util.cos_sim(query_emb, torch.tensor(cv_embeddings))[0]
top_k = torch.topk(cos_scores, k=3)

print("=== Top 3 Semantically Similar Resumes (KNN Retrieval) ===")
for score, idx in zip(top_k[0], top_k[1]):
    match = cv_metadata[idx.item()]
    print(f"- Resume #{match['id']} [{match['category']}] — Similarity: {score.item()*100:.1f}%")
    print(f"  Skills: {match['skills_preview']}")
    print(f"  Experience: {match['experience_preview']}\n")

=== Top 3 Semantically Similar Resumes (KNN Retrieval) ===
- Resume #1558 [ENGINEERING] — Similarity: 51.6%
  Skills: ['skills', 'creativity', 'Python Scripting', 'UNIX', 'Linux']
  Experience: ['TEST ENGINEERING', 'technology company', 'Test Engineer']

- Resume #1252 [DESIGNER] — Similarity: 47.0%
  Skills: ['technical skills', 'Unix', 'strong problem solving skills', 'C', 'NET']
  Experience: ['INFORMATION DESIGNER', '3 years', 'Chief technical staff']

- Resume #2065 [INFORMATION-TECHNOLOGY] — Similarity: 46.8%
  Skills: ['Linux', 'Apache2', 'HTTP', 'Python', 'Bash']
  Experience: ['INFORMATION TECHNOLOGY AND AWS ADMIN INTERN', 'AWS', 'Company Name']



## 4. Concept Separation: Similarity vs Career Suitability
> [!IMPORTANT]
> **Semantic Similarity** measures text co-occurrence and topical closeness against past candidates.  
> **Career Recommendation** requires strict verification of candidate technical skills, project evidence, and education requirements.  
> A resume can have 65% semantic overlap with a Consultant role while having 0% project or technical fit.
